# Image Segmentation with ClipSeg
This notebook uses the `CIDAS/clipseg-rd64-refined` model from Hugging Face to perform **text-based image segmentation**. You can provide text prompts, and the model will highlight the corresponding areas in the image.

In [ ]:
import os
import torch
import numpy as np
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from PIL import Image
import matplotlib.pyplot as plt

# --- FORCE STABILITY: Deep Device Check ---
def get_best_stable_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        dev = torch.device('cuda')
        dummy_input = torch.randn(1, 3, 32, 32).to(dev)
        dummy_model = torch.nn.Conv2d(3, 3, 3).to(dev)
        with torch.no_grad():
            dummy_model(dummy_input)
        print(f"CUDA is stable. Using: {torch.cuda.get_device_name(0)}")
        return 'cuda'
    except Exception as e:
        print(f"CUDA detected but incompatible ({e}). Falling back to CPU.")
        return 'cpu'

device = get_best_stable_device()

## Load Model and Processor

In [ ]:
model_id = "CIDAS/clipseg-rd64-refined"
print(f"Loading {model_id}...")

processor = CLIPSegProcessor.from_pretrained(model_id)
model = CLIPSegForImageSegmentation.from_pretrained(model_id).to(device)

print(f"Ready on {device}.")

## Segmentation Function
ClipSeg segments based on text prompts. We'll pass a list of things we want to find.

In [ ]:
def segment_image(image_path, prompts):
    image = Image.open(image_path).convert("RGB")
    
    # Pre-process image and prompts
    inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits are of shape (batch, 352, 352)
    preds = outputs.logits
    if len(prompts) == 1:
        preds = preds.unsqueeze(0)
        
    return image, preds

## Run Segmentation on Local Images
Change the `prompts` list below to look for different objects!

In [ ]:
image_folder = "../images"
image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

# Define what we want to segment
prompts = ["bird", "cat", "dog", "pizza", "cabinet", "cup"]

if not image_files:
    print("No images found.")
else:
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        print(f"\nProcessing: {img_file}")
        
        try:
            original_img, masks = segment_image(img_path, prompts)
            
            # Visualize
            n_prompts = len(prompts)
            fig, ax = plt.subplots(1, n_prompts + 1, figsize=(15, 5))
            
            # Show original
            ax[0].imshow(original_img)
            ax[0].set_title("Original")
            ax[0].axis("off")
            
            # Show each mask overlay
            for i in range(n_prompts):
                # Sigmoid to get probability-like values
                mask = torch.sigmoid(masks[i]).cpu().numpy()
                
                ax[i+1].imshow(original_img)
                ax[i+1].imshow(mask, alpha=0.6, cmap='jet') # Overlay with transparency
                ax[i+1].set_title(prompts[i])
                ax[i+1].axis("off")
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error processing {img_file}: {e}")